# River-basin runoff onset

HydroBASINS level-5 basins: the pixel-weighted median onset and MAD as basin choropleths, the annual anomalies by
basin, and two regional views (High Mountain Asia and western North America) with population, the share of
precipitation falling as snow and the major rivers.

| | |
| --- | --- |
| Reads | the river-basin cube and the per-basin metrics table written by `0_aggregate_by_river_basin.ipynb`; the BasinATLAS level-5 polygons (cached gdb); the hillshade basemap (`pixi run hillshade`); `results/<version>/river_basin_snow_water.csv` if `snow_water.ipynb` has run (the precipitation-as-snow panels stay empty otherwise); the World Bank *Major Rivers of the World*, read straight from the web |
| Writes | `figures/<version>/global_median_runoff_onset_and_mad.png`, `global_runoff_onset_anomaly_by_basin.png` and, for each region `hma` and `wus`: `<region>_runoff_onset_anomaly_by_basin.png`, `<region>_basin_population_and_pct_precip_as_snow.png`, `<region>_basin_runoff_onset_population_pct_precip_as_snow.png`, `<region>_basin_runoff_onset_vs_pct_precip_as_snow.png`, `<region>_basin_runoff_onset_vs_pct_precip_as_snow_largest_anom.png` |
| Needs | no credentials |

In [ ]:
import geopandas as gpd
import matplotlib
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray as rxr
import xarray as xr
from cartopy import crs as ccrs
from matplotlib.lines import Line2D

import easysnowdata
from gsro_analysis import paths, settings

In [ ]:
config = settings.load_config()  # the dataset version lives in settings.CONFIG_FILE

# the river-basin cube written by 0_aggregate_by_river_basin.ipynb: basin (HydroBASINS level 5) x elevation x chili_class x
# water_year, bin means as plain variables plus the per-basin ERA5-Land anomaly zonal means; fcf_lte_50 = the analyses' filter
river_basins_path = paths.aggregation_dir('river_basins', config.version) / 'all_river_basins_fcf_lte_50.nc'
river_basins_ds = xr.open_dataset(river_basins_path)
water_years = [int(y) for y in river_basins_ds['water_year'].values]
river_basins_ds

In [ ]:
# the per-basin metrics table written by 0_aggregate_by_river_basin.ipynb: pixel-weighted means of median onset / MAD and of
# the yearly onset and anomaly (masked where less than 5 % of the basin is mapped, yearly values where less than 1 %),
# pixel counts, the mapped share of the basin area, population
basin_metrics_path = paths.resultsdir('river_basins', config.version) / 'river_basin_metrics.csv'
basin_metrics_df = pd.read_csv(basin_metrics_path)
basin_metrics_df

In [ ]:
# HydroBASINS level-5 polygons from the cached BasinATLAS gdb (the cube's units; 2.7 GB, downloaded once), one row per
# PFAF_ID, joined to the metrics
basins_gdf = gpd.read_file(settings.basin_atlas_gdb(), layer=settings.basin_atlas_layer(5))[['PFAF_ID', 'SUB_AREA', 'geometry']]
if basins_gdf['PFAF_ID'].duplicated().any():
    basins_gdf = basins_gdf.dissolve(by='PFAF_ID', aggfunc={'SUB_AREA': 'sum'}, as_index=False)
basins_metrics_gdf = basins_gdf.merge(basin_metrics_df.drop(columns=['basin_area_km2']), on='PFAF_ID', how='inner')
basins_metrics_gdf

In [ ]:
# April-1 / October-1 SWE and the share of precipitation falling as snow per basin, from snow_water.ipynb (Earth Engine);
# the precipitation-as-snow panels stay empty when that table has not been written yet
snow_water_path = paths.resultsdir('river_basins', config.version) / 'river_basin_snow_water.csv'
if snow_water_path.exists():
    snow_water_df = pd.read_csv(snow_water_path)[['PFAF_ID', 'pct_precip_as_snow', 'swe_median']]
    basins_metrics_gdf = basins_metrics_gdf.merge(snow_water_df, on='PFAF_ID', how='left')
else:
    print(f'{snow_water_path} not found: run snow_water.ipynb first; the precipitation-as-snow panels will be empty')
    basins_metrics_gdf['pct_precip_as_snow'] = np.nan
    basins_metrics_gdf['swe_median'] = np.nan
basins_metrics_gdf[['PFAF_ID', 'runoff_onset_median', 'runoff_onset_mad', 'population', 'pct_precip_as_snow']].describe()

## Global maps

In [ ]:
# the hillshade basemap (pixi run hillshade), coarsened 10x for the global maps; row-strip chunks because the GeoTIFF is
# striped, so 'auto' 2-D chunks would make every dask thread decode the whole raster
hillshade_da = (rxr.open_rasterio(paths.hillshade(), masked=True, chunks={'y': 2048, 'x': -1}).squeeze()
                .coarsen(x=10, y=10, boundary='trim').mean().compute())
hillshade_da

In [ ]:
basins_metrics_robinson_gdf = basins_metrics_gdf.to_crs('ESRI:54030')
basins_metrics_robinson_gdf

In [ ]:
f, axs = plt.subplots(2, 1, figsize=(10, 8), subplot_kw={'projection': ccrs.Robinson()}, dpi=300, layout='constrained')

basins_metrics_robinson_gdf.plot(ax=axs[0], column='runoff_onset_median', cmap='viridis', legend=True,
                                 legend_kwds={'label': 'Median runoff onset date [DOWY]'}, transform=ccrs.Robinson(), vmin=110, vmax=250)
basins_metrics_robinson_gdf.plot(ax=axs[1], column='runoff_onset_mad', cmap='Reds', legend=True,
                                 legend_kwds={'label': 'Runoff onset MAD [days]'}, transform=ccrs.Robinson(), vmin=0, vmax=30)
for ax in axs:
    hillshade_da.plot.imshow(ax=ax, cmap='gray', transform=ccrs.Robinson(), zorder=0, add_colorbar=False)
    ax.gridlines(draw_labels=False, dms=True, x_inline=False, y_inline=False, xlocs=[-180, -120, -60, 0, 60, 120, 180],
                 ylocs=[-90, -60, -30, 0, 30, 60, 90], linestyle='--', linewidth=0.5)
    ax.set_title('')
axs[0].set_title(f'{len(water_years)}-year Median Snowmelt Runoff Onset by River Basin')
axs[1].set_title(f'{len(water_years)}-year Median Absolute Deviation of Snowmelt Runoff Onset by River Basin')

f.savefig(paths.figdir('river_basins', config.version) / 'global_median_runoff_onset_and_mad.png', bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
n_rows = -(-len(water_years) // 2)   # two columns, one panel per water year
f, axs = plt.subplots(nrows=n_rows, ncols=2, figsize=(10, 2.4 * n_rows), subplot_kw={'projection': ccrs.Robinson()}, layout='constrained')
for water_year, ax in zip(water_years, axs.flat):
    basins_metrics_robinson_gdf.plot(ax=ax, column=f'runoff_onset_anomaly_WY{water_year}', cmap='RdBu', vmin=-30, vmax=30, transform=ccrs.Robinson())
    hillshade_da.plot.imshow(ax=ax, cmap='gray', transform=ccrs.Robinson(), zorder=0, add_colorbar=False)
    ax.set_title(f'WY{water_year}')
    ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())
for ax in axs.flat[len(water_years):]:
    ax.set_visible(False)

f.savefig(paths.figdir('river_basins', config.version) / 'global_runoff_onset_anomaly_by_basin.png', bbox_inches='tight', dpi=300)

## Regional views: High Mountain Asia and western North America

Every cell below draws both regions. `REGIONS` holds the per-region map settings; `regional` the basins, hillshade and rivers
projected into each region's Albers equal-area CRS.

In [ ]:
REGIONS = {
    'hma': dict(title='High Mountain Asia',
                crs=ccrs.AlbersEqualArea(central_latitude=32.5, central_longitude=90),
                extent=[65, 108, 20, 50], hillshade_box=(55, 10, 115, 55), population_max=1e8,
                gridline_xlocs=[60, 70, 80, 90, 100, 110], gridline_ylocs=[20, 30, 40, 50],
                rivers=['Indus', 'Ganges', 'Brahmaputra', 'Yangtze', 'Mekong', 'Amu Darya', 'Syr Darya', 'Huang He'],
                legend_populations=[1e3, 1e4, 1e5, 1e6, 1e7, 1e8], legend_anchor=0.57),
    'wus': dict(title='Western North America',
                crs=ccrs.AlbersEqualArea(central_latitude=39, central_longitude=-120),
                extent=[-155, -105, 35, 73], hillshade_box=(-170, 25, -100, 80), population_max=1e7,
                gridline_xlocs=[-150, -140, -130, -120, -110], gridline_ylocs=[40, 50, 60, 70],
                rivers=['Columbia', 'Snake', 'Colorado', 'Missouri', 'Yukon', 'Mackenzie', 'Saskatchewan', 'Rio Grande, North America'],
                legend_populations=[1e3, 1e4, 1e5, 1e6, 1e7], legend_anchor=0.65),
}

In [ ]:
# the World Bank "Major Rivers of the World" (98 named rivers), read straight from the zip
major_rivers_gdf = gpd.read_file('zip+' + settings.MAJOR_RIVERS_URL)
major_rivers_gdf

In [ ]:
regional = {}
for region, cfg in REGIONS.items():
    basins_aea_gdf = basins_metrics_gdf.to_crs(cfg['crs'])
    regional[region] = dict(
        basins_gdf=basins_aea_gdf,
        # the basins inside the map window (the scatter plots and the precipitation-as-snow panels use these)
        roi_gdf=basins_aea_gdf.cx[-0.2e7:0.2e7, -0.35e7:0.35e7].copy(),
        hillshade_da=(hillshade_da.rio.clip_box(*cfg['hillshade_box'], crs='EPSG:4326').rio.reproject(cfg['crs'])
                      .coarsen(x=2, y=2, boundary='trim').mean()),
        rivers_gdf=major_rivers_gdf[major_rivers_gdf['NAME'].isin(cfg['rivers'])].to_crs(cfg['crs']),
    )
    print(f"{region}: {len(regional[region]['roi_gdf'])} basins in the window, {len(regional[region]['rivers_gdf'])} rivers")

In [ ]:
# annual anomalies by basin, one panel per water year
for region, cfg in REGIONS.items():
    n_cols = -(-len(water_years) // 2)
    f, axes = plt.subplots(nrows=2, ncols=n_cols, figsize=(3 * n_cols, 5.5), subplot_kw={'projection': cfg['crs']}, layout='constrained')
    for water_year, ax in zip(water_years, axes.flat):
        regional[region]['basins_gdf'].plot(ax=ax, column=f'runoff_onset_anomaly_WY{water_year}', cmap='RdBu', vmin=-30, vmax=30,
                                             transform=cfg['crs'], edgecolor='black', linewidth=0.2)
        regional[region]['hillshade_da'].plot.imshow(ax=ax, cmap='gray', transform=cfg['crs'], zorder=0, add_colorbar=False, vmin=0, vmax=255)
        ax.set_title(f'WY{water_year}')
        ax.set_extent(cfg['extent'], crs=ccrs.PlateCarree())
    for ax in axes.flat[len(water_years):]:
        ax.set_visible(False)
    norm = matplotlib.colors.Normalize(vmin=-30, vmax=30)
    sm = plt.cm.ScalarMappable(cmap='RdBu', norm=norm)
    sm.set_array([])
    cbar = f.colorbar(sm, ax=axes.ravel().tolist(), shrink=0.8, aspect=20, pad=0.02, extend='both')
    cbar.set_label('Snowmelt runoff onset anomaly [days]')
    f.suptitle(cfg['title'])
    f.savefig(paths.figdir('river_basins', config.version) / f'{region}_runoff_onset_anomaly_by_basin.png', bbox_inches='tight', dpi=300)

In [ ]:
# median onset and MAD per basin
for region, cfg in REGIONS.items():
    f, axs = plt.subplots(1, 2, figsize=(12, 4), subplot_kw={'projection': cfg['crs']}, dpi=300, layout='constrained')
    regional[region]['basins_gdf'].plot(ax=axs[0], column='runoff_onset_median', cmap='viridis', legend=True,
                                         legend_kwds={'label': 'Median runoff onset date [DOWY]'}, transform=cfg['crs'], vmin=110, vmax=250,
                                         edgecolor='black', linewidth=0.2)
    regional[region]['basins_gdf'].plot(ax=axs[1], column='runoff_onset_mad', cmap='Reds', legend=True,
                                         legend_kwds={'label': 'Runoff onset MAD [days]'}, transform=cfg['crs'], vmin=0, vmax=30,
                                         edgecolor='black', linewidth=0.2)
    for ax in axs:
        regional[region]['hillshade_da'].plot.imshow(ax=ax, cmap='gray', transform=cfg['crs'], zorder=0, add_colorbar=False, vmin=0, vmax=255)
        gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False, xlocs=cfg['gridline_xlocs'], ylocs=cfg['gridline_ylocs'],
                          linestyle='--', linewidth=0.5)
        gl.top_labels = False
        gl.right_labels = False
        ax.set_extent(cfg['extent'], crs=ccrs.PlateCarree())
    axs[0].set_title(f'{len(water_years)}-year Median Snowmelt Runoff Onset by River Basin')
    axs[1].set_title(f'{len(water_years)}-year Median Absolute Deviation of Snowmelt Runoff Onset by River Basin')
    f.suptitle(cfg['title'])

In [ ]:
# population and the share of precipitation falling as snow per basin
for region, cfg in REGIONS.items():
    f, axs = plt.subplots(1, 2, figsize=(10, 4), subplot_kw={'projection': cfg['crs']}, dpi=300, layout='constrained')
    regional[region]['basins_gdf'].plot(ax=axs[0], column='population', cmap='plasma', legend=True, legend_kwds={'label': 'Basin Population'},
                                         transform=cfg['crs'], norm=colors.LogNorm(vmin=100, vmax=cfg['population_max']), edgecolor='black', linewidth=0.2)
    regional[region]['roi_gdf'].plot(ax=axs[1], column='pct_precip_as_snow', cmap='Blues', legend=True, legend_kwds={'label': '%'},
                                      transform=cfg['crs'], edgecolor='black', linewidth=0.2, vmin=0, vmax=60)
    for ax in axs:
        regional[region]['hillshade_da'].plot.imshow(ax=ax, cmap='gray', transform=cfg['crs'], zorder=0, add_colorbar=False, vmin=0, vmax=255)
        gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False, xlocs=cfg['gridline_xlocs'], ylocs=cfg['gridline_ylocs'],
                          linestyle='--', linewidth=0.5)
        gl.top_labels = False
        gl.right_labels = False
        ax.set_extent(cfg['extent'], crs=ccrs.PlateCarree())
    axs[0].set_title('Basin Population')
    axs[1].set_title('Basin Percentage of Precipitation Falling as Snow')
    f.suptitle(cfg['title'])
    f.savefig(paths.figdir('river_basins', config.version) / f'{region}_basin_population_and_pct_precip_as_snow.png', bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
# the four basin views together, with the major rivers
for region, cfg in REGIONS.items():
    f, axs = plt.subplots(2, 2, figsize=(10, 7), subplot_kw={'projection': cfg['crs']}, dpi=300, layout='constrained')
    basins_aea_gdf = regional[region]['basins_gdf']
    basins_aea_gdf.plot(ax=axs[0, 0], column='runoff_onset_median', cmap='viridis', legend=True, legend_kwds={'label': 'Median runoff onset date [DOWY]'},
                        transform=cfg['crs'], vmin=110, vmax=250, edgecolor='black', linewidth=0.2, alpha=0.8)
    basins_aea_gdf.plot(ax=axs[0, 1], column='runoff_onset_mad', cmap='Reds', legend=True, legend_kwds={'label': 'Runoff onset MAD [days]'},
                        transform=cfg['crs'], vmin=0, vmax=30, edgecolor='black', linewidth=0.2, alpha=0.8)
    basins_aea_gdf.plot(ax=axs[1, 0], column='population', cmap='plasma', legend=True, legend_kwds={'label': 'Basin Population'},
                        transform=cfg['crs'], norm=colors.LogNorm(vmin=100, vmax=cfg['population_max']), edgecolor='black', linewidth=0.2, alpha=0.8)
    regional[region]['roi_gdf'].plot(ax=axs[1, 1], column='pct_precip_as_snow', cmap='Blues', legend=True, legend_kwds={'label': '%'},
                                      transform=cfg['crs'], edgecolor='black', linewidth=0.2, vmin=0, vmax=60, alpha=0.8)
    for ax in axs.flat:
        regional[region]['hillshade_da'].plot.imshow(ax=ax, cmap='gray', transform=cfg['crs'], zorder=0, add_colorbar=False, vmin=0, vmax=255)
        regional[region]['rivers_gdf'].plot(ax=ax, color='black', linewidth=1, zorder=2)
        ax.set_extent(cfg['extent'], crs=ccrs.PlateCarree())
    axs[0, 0].set_title(f'{len(water_years)}-year Median Snowmelt Runoff Onset')
    axs[0, 1].set_title(f'{len(water_years)}-year Median Absolute Deviation of Snowmelt Runoff Onset')
    axs[1, 0].set_title('Basin Population')
    axs[1, 1].set_title('Basin Percentage of Precipitation Falling as Snow')
    f.suptitle(cfg['title'])
    f.savefig(paths.figdir('river_basins', config.version) / f'{region}_basin_runoff_onset_population_pct_precip_as_snow.png', bbox_inches='tight', dpi=300, transparent=True)

### Onset date against the share of precipitation falling as snow

One dot per basin in the map window, its area scaling with population (radius × 1.5 per order of magnitude, 8 pt at 10⁴),
coloured by the basin's interannual variability (MAD) and, in the second figure, by the largest annual anomaly of the record.
Horizontal lines mark the first day of each month from January to June.

In [ ]:
# the largest (absolute) annual anomaly of each basin, with its sign
anomaly_cols = [f'runoff_onset_anomaly_WY{y}' for y in water_years]
for region in REGIONS:
    roi_gdf = regional[region]['roi_gdf']
    has_data = roi_gdf[anomaly_cols].notna().any(axis=1)
    roi_gdf['largest_anomaly_value'] = np.nan
    largest_col = roi_gdf.loc[has_data, anomaly_cols].abs().idxmax(axis=1)
    roi_gdf.loc[has_data, 'largest_anomaly_value'] = [roi_gdf.at[i, c] for i, c in largest_col.items()]
regional['hma']['roi_gdf'][['PFAF_ID', 'runoff_onset_median', 'pct_precip_as_snow', 'population', 'largest_anomaly_value']].describe()

In [ ]:
# month dividers on the day-of-water-year axis (a non-leap water year; Oct 1 = day 1)
MONTH_NAMES = ['October', 'November', 'December', 'January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September']
month_starts = ([easysnowdata.utils.datetime_to_DOWY(f'2021-{m:02d}-01') for m in range(10, 13)]
                + [easysnowdata.utils.datetime_to_DOWY(f'2022-{m:02d}-01') for m in range(1, 10)] + [366])
BASE_MARKERSIZE_PT = 8.0        # marker radius for a population of 1e4
RADIUS_PER_DECADE = 1.5         # radius multiplier per order of magnitude of population


def basin_scatter(ax, gdf, color_column, cmap, vmin, vmax, colorbar_label, legend_populations, legend_anchor):
    """Onset date vs precipitation-as-snow share, dots sized by population, coloured by ``color_column``."""
    magnitude_offset = np.log10(gdf['population'].clip(lower=1)) - 4.0
    marker_radii = BASE_MARKERSIZE_PT * (RADIUS_PER_DECADE ** magnitude_offset)
    sc = ax.scatter(gdf['pct_precip_as_snow'], gdf['runoff_onset_median'], s=marker_radii ** 2, c=gdf[color_column],
                    cmap=cmap, vmin=vmin, vmax=vmax, alpha=0.8, edgecolors='k', linewidth=0.5)
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label(colorbar_label)
    for month, start, end in zip(MONTH_NAMES, month_starts[:-1], month_starts[1:]):
        if month in ['January', 'February', 'March', 'April', 'May', 'June']:
            ax.axhline(y=start, color='black', linestyle='--', linewidth=0.5, alpha=0.5, zorder=1)
            ax.text(6, (start + end) / 2, month, rotation=90, va='center', ha='center', fontsize=9, color='black')
    legend_radii = BASE_MARKERSIZE_PT * (RADIUS_PER_DECADE ** (np.log10(np.array(legend_populations)) - 4.0))
    legend_labels = [f'{int(p / 1e6)}M' if p >= 1e6 else f'{int(p / 1e3)}K' for p in legend_populations]
    handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=radius, label=label,
                      markeredgecolor='k', markeredgewidth=0.5) for radius, label in zip(legend_radii, legend_labels)]
    ax.legend(handles=handles, title='Basin population', loc='lower center', bbox_to_anchor=(legend_anchor, 0.003),
              ncol=len(handles), framealpha=1, columnspacing=1.3, handleheight=3.5, handletextpad=1)
    ax.set_xlabel('Percentage of precipitation falling as snow [%]')
    ax.set_ylabel('Median snowmelt runoff onset date [DOWY]')
    ax.set_ylim(100, 265)
    ax.set_xlim(5, 75)
    return sc

In [ ]:
for region, cfg in REGIONS.items():
    f, ax = plt.subplots(figsize=(8, 6), dpi=300, layout='constrained')
    basin_scatter(ax, regional[region]['roi_gdf'], 'runoff_onset_mad', 'Reds', 5, 30,
                  f'{len(water_years)}-yr runoff onset MAD [days]', cfg['legend_populations'], cfg['legend_anchor'])
    ax.set_title(f"{cfg['title']} river basins: interannual variability")
    f.savefig(paths.figdir('river_basins', config.version) / f'{region}_basin_runoff_onset_vs_pct_precip_as_snow.png', bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
for region, cfg in REGIONS.items():
    f, ax = plt.subplots(figsize=(8, 6), dpi=300, layout='constrained')
    basin_scatter(ax, regional[region]['roi_gdf'], 'largest_anomaly_value', 'RdBu', -30, 30,
                  f'Largest runoff onset anomaly WY{water_years[0]}-{water_years[-1]} [days]', cfg['legend_populations'], cfg['legend_anchor'])
    ax.set_title(f"{cfg['title']} river basins: largest runoff onset anomaly WY{water_years[0]}-{water_years[-1]}")
    f.savefig(paths.figdir('river_basins', config.version) / f'{region}_basin_runoff_onset_vs_pct_precip_as_snow_largest_anom.png', bbox_inches='tight', dpi=300, transparent=True)